In [ ]:
import python_calamine
import pandas as pd
import numpy as np
from pathlib import Path
import xlsxwriter

In [ ]:
base_path = Path.cwd()
raw_data = base_path / "data" / "raw_data.xlsx"

raw_data_excel = pd.read_excel(raw_data, sheet_name="CAPWindModeling")
# raw_data_df.head(10)
raw_data_df = raw_data_excel.drop(columns=["Unnamed: 38"]).copy()

In [ ]:
raw_data_df.head()

In [ ]:
RENAME_COLS = {
    'MCONAME' : 'MCONAME',
    'Agent Name' : 'Agent Name',
    'Named Insured' : 'Named Insured',  #get ACCNTNAME from name insured with commas stripped for proper CSV parsing
    'QUOTEID' : 'QUOTEID',  # ACCNTNUM gets value from this column
    'LOCNO' : 'LOCNO',
    'BLDGNO' : 'BLDGNO',    #get LOCNUM column from this column
    'STNAME' : 'STNAME',    #get STREETNAME column from this column
    'CITY' : 'CITY',        #duplicate CITY column? --this probably matters when we get only the yellow columns for the CSV output so that the CSV parses correctly when inputted into the model
    'STATE' : 'STATE',      #get STATECODE column from this column
    'ZIP5' : 'ZIP5',        #get POSTALCODE column from this column
    'COUNTY' : 'COUNTY',    #duplicate COUNTY column; other columns: CNTRYSCHEME == ISO2A, CNTRYCODE == US
    'CONSTCL' : 'CONSTCL',  #after this we have BLDGSCHEME == FIRE and BLDGCLASS which comes from this column
    'OCCPCL' : 'OCCPCL',    #after this we have OCCSCHEME == ATC and OCCTYPE which comes from this column
    'NOSTORIES' : 'NOSTORIES',  #get NUMSTORIES column from this column
    'YEARBUILT' : 'YEARBUILT',  #duplicate YEARBUILT column
    'RMSLOB' : 'RMSLOB',
    'WINDPCTDED' : 'WINDPCTDED',
    'LOCBLKLIMIT' : 'LOCBLKLIMIT',
    'LOCBLKDEDU' : 'LOCBLKDEDU',
    'LOCBLDREPL' : 'LOCBLDREPL',    #get CV1VAL column from this column
    'LOCBLDLIMT' : 'LOCBLDLIMT',
    'LOCBLDDEDU' : '*LOCBLDDEDU',
    'LOCCNTREPL' : 'LOCCNTREPL',    #get CV2VAL column from this column
    'COCCNTLIMT' : 'COCCNTLIMT',
    'LOCCNTDEDU' : 'LOCCNTDEDU',
    'LOCBIRC' : 'LOCBIRC',
    'LOCBILIMIT' : 'LOCBILIMIT',
    'LOCBIDEDU' : 'LOCBIDEDU',
    'LOCBLDPREMAOP' : 'LOCBLDPREMAOP',
    'LOCBLDPREMWIND' : 'LOCBLDPREMWIND',
    'LOCCNTPREMAOP' : 'LOCCNTPREMAOP',
    'LOCCNTPREMWIND' : 'LOCCNTPREMWIND',
    'LOCBIPREM' : 'LOCBIPREM',
    'SQFEET' : 'SQFEET',    #get FLOORAREA column from this column
    'RATINGTERR' : 'RATINGTERR',
    'SPRINKLER' : 'SPRINKLER',
    'PROTCLASS' : 'PROTCLASS',
    'DESCRIPTION' : 'DESCRIPTION',
    'Effective Date' : 'Effective Date',    #get INCEPTDATE column from this column
    'Expiration Date' : 'Expiration Date',  #get EXPIREDATE column from this column
    'CONSTQUA' : '*CONSTQUALI',
    'ROOFSYS' : '*ROOFSYS',
    'ROOFAGE' : '*ROOFAGE',
    'ROOFGEO' : '*ROOFGEOM',
    'ROOFANC' : '*ROOFANCH',
    'CLADSYS' : '*CLADSYS',
    'FOUNDSYS' : '*FOUNDSYS',
    'ROOFEQUI' : '*ROOFEQUIP',
    'CLADRATE' : '*CLADRATE',
    'RESISTOPEN' : '*RESISTOPEN'
}

SCENARIOS = [
    ('2pct', 'SCS', 3, 0.02),
    ('2pct', 'WS', 2, 0.02),
    ('3pct', 'WS', 2, 0.03),
    ('5pct', 'WS', 2, 0.05)
]

In [ ]:
raw_data_df = raw_data_df.rename(columns = RENAME_COLS)

raw_data_df.insert(raw_data_df.columns.get_loc('Named Insured') + 1, '*ACCNTNAME', raw_data_df['Named Insured'].str.replace(',', ''))
raw_data_df.insert(raw_data_df.columns.get_loc('QUOTEID') + 1, '*ACCNTNUM', raw_data_df['QUOTEID'])
raw_data_df.insert(raw_data_df.columns.get_loc('BLDGNO') + 1, '*LOCNUM', raw_data_df['BLDGNO'])
raw_data_df.insert(raw_data_df.columns.get_loc('COUNTY') + 1, '*STREETNAME', raw_data_df['STNAME'])
raw_data_df.insert(raw_data_df.columns.get_loc('COUNTY') + 2, '*CITY', raw_data_df['CITY'])
raw_data_df.insert(raw_data_df.columns.get_loc('COUNTY') + 3, '*STATECODE', raw_data_df['STATE'])
raw_data_df.insert(raw_data_df.columns.get_loc('COUNTY') + 4, '*POSTALCODE', raw_data_df['ZIP5'])
raw_data_df.insert(raw_data_df.columns.get_loc('COUNTY') + 5, '*COUNTY', raw_data_df['COUNTY'])
raw_data_df.insert(raw_data_df.columns.get_loc('COUNTY') + 6, '*CNTRYSCHEME', 'ISO2A')
raw_data_df.insert(raw_data_df.columns.get_loc('COUNTY') + 7, '*CNTRYCODE', 'US')
raw_data_df.insert(raw_data_df.columns.get_loc('CONSTCL') + 1, '*BLDGSCHEME', 'FIRE')
raw_data_df.insert(raw_data_df.columns.get_loc('CONSTCL') + 2, '*BLDGCLASS', raw_data_df['CONSTCL'])
raw_data_df.insert(raw_data_df.columns.get_loc('OCCPCL') + 1, '*OCCSCHEME', 'ATC')
raw_data_df.insert(raw_data_df.columns.get_loc('OCCPCL') + 2, '*OCCTYPE', raw_data_df['OCCPCL'])
raw_data_df.insert(raw_data_df.columns.get_loc('NOSTORIES') + 1, '*NUMSTORIES', raw_data_df['NOSTORIES'])
raw_data_df.insert(raw_data_df.columns.get_loc('YEARBUILT') + 1, '*YEARBUILT', '1/1/' + raw_data_df['YEARBUILT'].astype(str))
raw_data_df.insert(raw_data_df.columns.get_loc('LOCBLDREPL'), '*CV1VAL', raw_data_df['LOCBLDREPL'])
raw_data_df.insert(raw_data_df.columns.get_loc('LOCCNTREPL'), '*CV2VAL', raw_data_df['LOCCNTREPL'])
raw_data_df.insert(raw_data_df.columns.get_loc('SQFEET') + 1, '*FLOORAREA', raw_data_df['SQFEET'])
raw_data_df.insert(raw_data_df.columns.get_loc('Expiration Date') + 1, '*INCEPTDATE', raw_data_df['Effective Date'])
raw_data_df.insert(raw_data_df.columns.get_loc('Expiration Date') + 2, '*EXPIREDATE', raw_data_df['Expiration Date'])


In [ ]:
acct_temp = (raw_data_df[['*ACCNTNUM','*ACCNTNAME','*INCEPTDATE','*EXPIREDATE','*LOCBLDDEDU']]
              .drop_duplicates(subset = ['*ACCNTNUM']))

location_temp = raw_data_df[[col for col in raw_data_df.columns if col.startswith('*')]]

acct_scenario_dfs = []
loc_scenario_dfs = []

for scenario in SCENARIOS:
    pct, pnum, ptype, prob = scenario

    acct_scenario_df = acct_temp.copy()
    acct_scenario_df['*ACCNTNUM'] = acct_scenario_df['*ACCNTNUM'].astype(str) + '_' + pct
    acct_scenario_df.insert(acct_scenario_df.columns.get_loc('*EXPIREDATE') + 1, '*POLICYNUM', pnum)
    acct_scenario_df.insert(acct_scenario_df.columns.get_loc('*EXPIREDATE') + 2, '*POLICYTYPE', ptype)

    loc_scenario_df = location_temp.copy()
    loc_scenario_df['*ACCNTNUM'] = loc_scenario_df['*ACCNTNUM'].astype(str) + '_' + pct
    loc_scenario_df.insert(loc_scenario_df.columns.get_loc('*CV1VAL') + 1, '*WSSITEDED', prob)

    if pnum == 'SCS':
        acct_scenario_df['*BLANDEDAMT'] = acct_scenario_df['*LOCBLDDEDU']
        acct_scenario_df['*MINDEDAMT'] = np.nan

        loc_scenario_df.insert(loc_scenario_df.columns.get_loc('*CV2VAL') + 1, '*TOCV1VAL', loc_scenario_df['*CV1VAL'])
        loc_scenario_df.insert(loc_scenario_df.columns.get_loc('*CV2VAL') + 2, '*TOCV2VAL', loc_scenario_df['*CV2VAL'])
        loc_scenario_df.insert(loc_scenario_df.columns.get_loc('*CV1VAL') + 1, '*WSCV1VAL', np.nan)
        loc_scenario_df.insert(loc_scenario_df.columns.get_loc('*CV1VAL') + 2, '*WSCV2VAL', np.nan)
        
    else:
        acct_scenario_df['*BLANDEDAMT'] = np.nan
        acct_scenario_df['*MINDEDAMT'] = acct_scenario_df['*LOCBLDDEDU']

        loc_scenario_df.insert(loc_scenario_df.columns.get_loc('*CV2VAL') + 1, '*TOCV1VAL', np.nan)
        loc_scenario_df.insert(loc_scenario_df.columns.get_loc('*CV2VAL') + 2, '*TOCV2VAL', np.nan)
        loc_scenario_df.insert(loc_scenario_df.columns.get_loc('*CV1VAL') + 1, '*WSCV1VAL', loc_scenario_df['*CV1VAL'])
        loc_scenario_df.insert(loc_scenario_df.columns.get_loc('*CV1VAL') + 2, '*WSCV2VAL', loc_scenario_df['*CV2VAL'])

    loc_scenario_df = (loc_scenario_df
                       .drop(columns=['*CV1VAL', '*CV2VAL']))

    acct_scenario_dfs.append(acct_scenario_df)
    loc_scenario_dfs.append(loc_scenario_df)

account_df = pd.concat(acct_scenario_dfs, ignore_index=True).drop(columns=['*LOCBLDDEDU'])

location_df = pd.concat(loc_scenario_dfs, ignore_index=True).drop(columns=['*LOCBLDDEDU'])
location_df = (location_df
               .groupby(['*ACCNTNUM', '*LOCNUM'], as_index=False, sort=False)
               .first())